## 01 Asistente de email - config

In [ ]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [ ]:
profile = {
    "name": "Sarah",
    "full_name": "Sarah Chen",
    "user_profile_background": "Ingeniera de software senior liderando un equipo de 5 desarrolladores",
}

Definiendo instrucciones de prompt


In [ ]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Newsletters de marketing, e-mails de spam, comunicados generales de la empresa",
        "notify": "Miembro del equipo convaleciente, notificaciones del sistema d build, Actualizaciones del status del proyecto",
        "respond": "Preguntas directas de miembros del equipo, solicitudes de reunión, informes de bugs críticos",
    },
    "agent_instructions": "Usa estas heerramientas cuando sea apropiado para ayudar a gestionar las tareas de Sarah de forma eficiente."
}

Presentando un ejemplo de correo electrónico


In [ ]:
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Duda rápida sobre la documentación de la API",
    "body": """
Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice""",
}

Realizando importaciones necesarias


In [ ]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

Creando la clase Router


In [ ]:
class Router(BaseModel):
    """Analiza el e-mail no leído y enrrútalo de acuerdo con su contenido."""

    reasoning: str = Field(
        description="Raciocinio paso a paso tras la clasificación."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="La clasificación de un e-mail: `ignore` para e-mails irrelevantes, "
        "`notify` para informaciones importantes que no necesitan de una respuesta, "
        "`respond` para e-mails que requieren una respuesta",
    )

Implementando la clase y configurando prompts


In [ ]:
llm_router = llm.with_structured_output(Router)

In [ ]:
from prompts import triage_system_prompt, triage_user_prompt

#### Utilizando tags y memoria semántica
Utilizamos tags para organizar las funciones. Por ejemplo, después de la función, viene otro tag que sería instrucciones. También manejamos una memoria semántica del agent system prompt, que se gestionará más adelante.

#### Configurando prompts de triaje
Tenemos un prompt de triaje con instrucciones claras: ignorar, notificar y responder. Estas son las clasificaciones:

Triaje ignorar: Lo que no vale la pena responder.\
Triaje notificar: Lo que debe ser notificado.\
Triaje responder: Correos que requieren respuesta.\
Se proporcionan ejemplos y también tenemos el triaje user prompt, que determina cómo manejar la conversación de correo electrónico, incluyendo el autor, destinatario y asunto del hilo del email.

Importando y creando prompts


In [ ]:
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

In [ ]:
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"],
)

Invocando el roteador DLLM

In [ ]:
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

In [ ]:
print(result)

## 02 Asistente de email - herramientas

In [ ]:
%pip install -qU langgraph-prebuilt

In [ ]:
from langchain_core.tools import tool

Definiendo herramientas para correos y reuniones


In [ ]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Escribe y envia un e-mail."""
    # Respuesta de placeholder - en un aplicativo real, enviaria el e-mail
    return f"E-mail enviado para {to} con el asunto {subject}"

In [ ]:
@tool
def schedule_meeting(
    attendees: list[str],
    subject: str,
    duration_minutes: int,
    preferred_day: str
) -> str:
    """Programa una reunión en el calendario."""
    return f"Reunión '{subject}' programada para {preferred_day} con {len(attendees)} participantes"

Verificando disponibilidad del calendario


In [ ]:
@tool
def check_calendar_availability(day: str) -> str:
    """Verifica la disponibilidad de agenda para una fecha determinada."""
    return f"Horarios disponibles en {day}: 9:00 AM, 2:00 PM, 4:00 PM"

Creando el prompt del sistema del agente


In [ ]:
from prompts import agent_system_prompt

def create_prompt(state):
    return [
        {
            "role": "system",
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
            ),
        }
    ] + state['messages']

In [ ]:
print(agent_system_prompt)

Creando el agente React


In [ ]:
from langgraph.prebuilt import create_react_agent

tools=[write_email, schedule_meeting, check_calendar_availability]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=create_prompt,
)

Ejecutando y probando el agente


In [ ]:
response = agent.invoke(
    {"messages": [
        {
            "role": "user",
            "content": "Cuál es mi disponibilidad para el martes?"
        }
    ]}
)

In [ ]:
response["messages"][-1].pretty_print()

Creando el estado del agente


In [ ]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

Definiendo el enrutador de triaje


In [ ]:
def triage_router(state: State) -> Command[
    Literal["response_agent", "__end__"]
]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no_prompt_instructions=triage_rules["ignore"],
        triage_notify_prompt_instructions=triage_rules["notify"],
        triage_email_prompt_instructions=triage_rules["respond"],
        examples=None
    )

    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    if result.classification == "respond":
        print(f"�� Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print(f"�� Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = END
    elif result.classification == "notify":
        # En un escenario real, se realizaria otra acción
        print(f"�� Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = END
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")

    return Command(goto=goto, update=update)

## 03 Asistente de email - ejecución

In [ ]:
email_agent = StateGraph(State)

Añadiendo nodos y aristas al grafo


In [ ]:
email_agent = email_agent.add_node("triage_router", triage_router)
email_agent = email_agent.add_node("response_agent", agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile()

In [ ]:
display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

Creando y clasificando correos electrónicos


In [ ]:
email_input = {
    "author":"Equipo de Marketing <marketing@amazingdeals.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "�� OFERTA EXCLUSIVA: ¡Descuento por Tiempo Limitado en Herramientas para Desarrolladores!",
    "email_thread": "Estimado(a) Desarrollador(a),\n\n¡No pierda esta oportunidad INCREÍBLE!\n\n�� POR TIEMPO LIMITADO, obtenga un 80% DE DESCUENTO en nuestro Paquete Premium para Desarrolladores\n\n�� RECURSOS:\n- Autocompletado de código revolucionario con IA\n- Entorno de desarrollo basado en la nube\n- Soporte al cliente 24/7\n- ¡Y mucho más!\n\n�� Precio normal: R$ 999/mes\n�� SU PRECIO ESPECIAL: ¡Solo R$ 199/mes!\n\n⏳ ¡Apúrese! Esta oferta expira en:\n¡SOLO 24 HORAS!\n\nHaga clic aquí para canjear su descuento: https://amazingdeals.com/special-offer\n\nAtentamente,\nEquipo de Marketing\n\nPara cancelar la suscripción, haga clic aquí",
}

In [ ]:
response = email_agent.invoke({"email_input": email_input})

Respondiendo a correos electrónicos internos


In [ ]:
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,\n\nEstaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints no están claros.\n\nEspecíficamente, estoy buscando:\n- /auth/refresh\n- /auth/validate\n\n¡Gracias!\nAlice""",
}

In [ ]:
response = email_agent.invoke({"email_input": email_input})

In [ ]:
for m in response["messages"]:
    m.pretty_print()